
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Build the Model (RAG Chain)

This notebook should be run before the module to create and register a model that will be used for the rest of the module.

**🚨 Note**: This notebook is run by `Workspace Setup`. You don't need to manually run this notebook.

### Classroom Setup

In [0]:
%pip install -qq -U databricks-vectorsearch llama-index PyPDF2 databricks-sdk langchain==0.3.7 langchain-community==0.3.7 langchain-databricks

dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
petastorm 0.12.1 requires pyspark>=2.1.0, which is not installed.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.11.0 which is incompatible.
numba 0.57.1 requires numpy<1.25,>=1.21, but you have numpy 1.26.4 which is incompatible.
ydata-profiling 4.5.1 requires numpy<1.24,>=1.16.0, but you have numpy 1.26.4 which is incompatible.
ydata-profiling 4.5.1 requires pydantic<2,>=1.8.1, but you have pydantic 2.12.4 which is incompatible.
virtualenv 20.24.2 requires platformdirs<4,>=3.9.1, but you have platformdirs 4.5.0 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run ../Includes/Classroom-Setup-00

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.



The examples and models presented in this course are intended solely for demonstration and educational purposes.
 Please note that the models and prompt examples may sometimes contain offensive, inaccurate, biased, or harmful content.


## Create Shared Catalog

In [0]:
catalog_name = "genai_shared_catalog_04"
schema_name = f"ws_{spark.conf.get('spark.databricks.clusterUsageTags.clusterOwnerOrgId')}"

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalog_name}`;")
spark.sql(f"USE CATALOG `{catalog_name}`;")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{schema_name}`;")
spark.sql(f"USE SCHEMA `{schema_name}`;")
spark.sql(f"GRANT USE CATALOG, USE SCHEMA, EXECUTE ON CATALOG `{catalog_name}` TO `account users`;")

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-5151829411170568>, line 1
----> 1 spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalog_name}`;")
      2 spark.sql(f"USE CATALOG `{catalog_name}`;")
      3 spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{schema_name}`;")

File /databricks/spark/python/pyspark/instrumentation_utils.py:47, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     45 start = time.perf_counter()
     46 try:
---> 47     res = func(*args, **kwargs)
     48     logger.log_success(
     49         module_name, class_name, function_name, time.perf_counter() - start, signature
     50     )
     51     return res

File /databricks/spark/python/pyspark/sql/session.py:1830, in SparkSession.sql(self, sqlQuery, args, **kwargs)
   1825     else:
   1826         raise PySparkTypeError(
   1827             error_class="INVALID_TYPE",
   1828             message_p

## Helpers

In [0]:
import time
import re
import io
import os
import pandas as pd 

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document
from llama_index.core.utils import set_global_tokenizer
from transformers import AutoTokenizer
from typing import Iterator
from pyspark.sql.functions import col, udf, length, pandas_udf, explode
from PyPDF2 import PdfReader

2025-11-15 22:03:25.213252: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-15 22:03:25.217198: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-15 22:03:25.270259: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-15 22:03:28.650092: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [0]:
def parse_bytes_pypdf(raw_doc_contents_bytes: bytes):
    try:
        pdf = io.BytesIO(raw_doc_contents_bytes)
        reader = PdfReader(pdf)
        parsed_content = [page_content.extract_text() for page_content in reader.pages]
        return "\n".join(parsed_content)
    except Exception as e:
        warnings.warn(f"Exception {e} has been thrown during parsing")
        return None 
      
def extract_doc_text(x : bytes) -> str:
  # Read files and extract the values with unstructured
  sections = partition(file=io.BytesIO(x))
  def clean_section(txt):
    txt = re.sub(r'\n', '', txt)
    return re.sub(r' ?\.', '.', txt)
  # Default split is by section of document
  # concatenate them all together because we want to split by sentence instead.
  return "\n".join([clean_section(s.text) for s in sections]) 


def pprint(obj):
  import pprint
  pprint.pprint(obj, compact=True, indent=1, width=100)

def index_exists(vsc, endpoint_name, index_full_name):
  try:
      dict_vsindex = vsc.get_index(endpoint_name, index_full_name).describe()
      return dict_vsindex.get('status').get('ready', False)
  except Exception as e:
      if 'RESOURCE_DOES_NOT_EXIST' not in str(e):
          print(f'Unexpected error describing the index. This could be a permission issue.')
          raise e
  return False

def wait_for_vs_endpoint_to_be_ready(vsc, vs_endpoint_name):
  for i in range(180):
    endpoint = vsc.get_endpoint(vs_endpoint_name)
    status = endpoint.get("endpoint_status", endpoint.get("status"))["state"].upper()
    if "ONLINE" in status:
      return endpoint
    elif "PROVISIONING" in status or i <6:
      if i % 20 == 0: 
        print(f"Waiting for endpoint to be ready, this can take a few min... {endpoint}")
      time.sleep(10)
    else:
      raise Exception(f'''Error with the endpoint {vs_endpoint_name}. - this shouldn't happen: {endpoint}.\n Please delete it and re-run the previous cell: vsc.delete_endpoint("{vs_endpoint_name}")''')
  raise Exception(f"Timeout, your endpoint isn't ready yet: {vsc.get_endpoint(vs_endpoint_name)}")

def wait_for_index_to_be_ready(vsc, vs_endpoint_name, index_name):
  for i in range(180):
    idx = vsc.get_index(vs_endpoint_name, index_name).describe()
    index_status = idx.get('status', idx.get('index_status', {}))
    status = index_status.get('detailed_state', index_status.get('status', 'UNKNOWN')).upper()
    url = index_status.get('index_url', index_status.get('url', 'UNKNOWN'))
    if "ONLINE" in status:
      return
    if "UNKNOWN" in status:
      print(f"Can't get the status - will assume index is ready {idx} - url: {url}")
      return
    elif "PROVISIONING" in status:
      if i % 40 == 0: print(f"Waiting for index to be ready, this can take a few min... {index_status} - pipeline url:{url}")
      time.sleep(10)
    else:
        raise Exception(f'''Error with the index - this shouldn't happen. DLT pipeline might have been killed.\n Please delete it and re-run the previous cell: vsc.delete_index("{index_name}, {vs_endpoint_name}") \nIndex details: {idx}''')
  raise Exception(f"Timeout, your index isn't ready yet: {vsc.get_index(index_name, vs_endpoint_name)}")

## Prepare Data

1. Read raw `pdf` into `pdf_raw_text` delta table
2. Chunk using llama-index & pandas_udf
3. Materialize chunked documents into `pdf_text_chunks` delta table

In [0]:
# Reduce the arrow batch size as our PDF can be big in memory
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", 10)

articles_path = f"{DA.paths.datasets.arxiv}/arxiv-articles/"
table_name = f"{catalog_name}.{schema_name}.pdf_raw_text"

# read pdf files
df = (
        spark.read.format("binaryfile")
        .option("recursiveFileLookup", "true")
        .load(articles_path)
        )

# save list of the files to table
df.write.mode("overwrite").saveAsTable(table_name)

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-5151829411170573>, line 15
      8 df = (
      9         spark.read.format("binaryfile")
     10         .option("recursiveFileLookup", "true")
     11         .load(articles_path)
     12         )
     14 # save list of the files to table
---> 15 df.write.mode("overwrite").saveAsTable(table_name)

File /databricks/spark/python/pyspark/instrumentation_utils.py:47, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     45 start = time.perf_counter()
     46 try:
---> 47     res = func(*args, **kwargs)
     48     logger.log_success(
     49         module_name, class_name, function_name, time.perf_counter() - start, signature
     50     )
     51     return res

File /databricks/spark/python/pyspark/sql/readwriter.py:1855, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
   1853 if forma

In [0]:
@pandas_udf("array<string>")
def read_as_chunk(batch_iter: Iterator[pd.Series]) -> Iterator[pd.Series]:
    #set llama2 as tokenizer
    set_global_tokenizer(
      AutoTokenizer.from_pretrained("hf-internal-testing/llama-tokenizer")
    )
    #Sentence splitter from llama_index to split on sentences
    splitter = SentenceSplitter(chunk_size=500, chunk_overlap=50)
    def extract_and_split(b):
      txt = parse_bytes_pypdf(b) 
      nodes = splitter.get_nodes_from_documents([Document(text=txt)])
      return [n.text for n in nodes]

    for x in batch_iter:
        yield x.apply(extract_and_split)

In [0]:
df_chunks = (df
                .withColumn("content", explode(read_as_chunk("content")))
                .selectExpr('path as pdf_name', 'content')
                )

In [0]:
%sql
CREATE TABLE IF NOT EXISTS pdf_text_chunks (
  id BIGINT GENERATED BY DEFAULT AS IDENTITY,
  pdf_name STRING,
  content STRING,
  embedding ARRAY <FLOAT>
  ) TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
chunks_table_name = f"{catalog_name}.{schema_name}.pdf_text_chunks"
df_chunks.write.mode("append").saveAsTable(chunks_table_name)

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-5151829411170577>, line 2
      1 chunks_table_name = f"{catalog_name}.{schema_name}.pdf_text_chunks"
----> 2 df_chunks.write.mode("append").saveAsTable(chunks_table_name)

File /databricks/spark/python/pyspark/instrumentation_utils.py:47, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     45 start = time.perf_counter()
     46 try:
---> 47     res = func(*args, **kwargs)
     48     logger.log_success(
     49         module_name, class_name, function_name, time.perf_counter() - start, signature
     50     )
     51     return res

File /databricks/spark/python/pyspark/sql/readwriter.py:1855, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
   1853 if format is not None:
   1854     self.format(format)
-> 1855 self._jwrite.saveAsTable(name)

File /databricks/spark/python/lib/py4j-0.1

## Create Databricks Vector Search Endpoint

1. Spin-up VS endpoint
2. Create index from `pdf_text_chunks` delta table

In [0]:
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient
import databricks.sdk.service.catalog as c


vs_endpoint_name = "genai_vs_endpoint"
source_table_fullname = f"{catalog_name}.{schema_name}.pdf_text_chunks"
vs_index_fullname = f"{catalog_name}.{schema_name}.pdf_text_vs_index"

vsc = VectorSearchClient(disable_notice=True)


if vs_endpoint_name not in [e['name'] for e in vsc.list_endpoints().get('endpoints', [])]:
    vsc.create_endpoint(
        name=vs_endpoint_name, 
        endpoint_type="STANDARD",
        )

wait_for_vs_endpoint_to_be_ready(vsc, vs_endpoint_name)
print(f"Endpoint named {vs_endpoint_name} is ready.")

# create or sync the index
if not index_exists(vsc, vs_endpoint_name, vs_index_fullname):
  print(f"Creating index {vs_index_fullname} on endpoint {vs_endpoint_name}...")
  vsc.create_delta_sync_index(
    endpoint_name=vs_endpoint_name,
    index_name=vs_index_fullname,
    source_table_name=source_table_fullname,
    pipeline_type="TRIGGERED",
    primary_key="id",
    embedding_dimension=1024, #Match your model embedding size (bge)
    embedding_source_column="content",
    embedding_model_endpoint_name="databricks-gte-large-en"
  )
else:
  #Trigger a sync to update our vs content with the new data saved in the table
  vsc.get_index(vs_endpoint_name, vs_index_fullname).sync()

#Let's wait for the index to be ready and all our embeddings to be created and indexed
wait_for_index_to_be_ready(vsc, vs_endpoint_name, vs_index_fullname)

---------------------------------------------------------------------------
HTTPError                                 Traceback (most recent call last)
File /local_disk0/.ephemeral_nfs/envs/pythonEnv-1cfa57ef-7285-4082-ac52-5d2aced5a9e1/lib/python3.11/site-packages/databricks/vector_search/utils.py:78, in RequestUtils.issue_request(url, method, token, params, json, verify, auth, data, headers)
     77 try:
---> 78     response.raise_for_status()
     79 except Exception as e:

File /databricks/python/lib/python3.11/site-packages/requests/models.py:1021, in Response.raise_for_status(self)
   1020 if http_error_msg:
-> 1021     raise HTTPError(http_error_msg, response=self)

HTTPError: 403 Client Error: Forbidden for url: https://oregon.cloud.databricks.com/api/2.0/vector-search/endpoints/genai_vs_endpoint

During handling of the above exception, another exception occurred:

Exception                                 Traceback (most recent call last)
File <command-5151829411170579>, line 

### Grant `SELECT` access to both catalog and created index to current users & `Account Users`
For demo purposes only

In [0]:
spark.sql(f"GRANT `USE SCHEMA` ON SCHEMA {catalog_name}.{schema_name} TO `account users`;")
spark.sql(f"GRANT SELECT ON TABLE {vs_index_fullname} TO `account users`;")

# spark.sql(f"GRANT `USE SCHEMA` ON SCHEMA {catalog_name}.{schema_name} TO `{DA.username}`;")
# spark.sql(f"GRANT SELECT ON TABLE {vs_index_fullname} TO `{DA.username}`;")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:445)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:464)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:571)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:528)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:633)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:656)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:48)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:276)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:62)
	at com.databricks.logging.AttributionContext$.withValue(Attr

## Create the Chain


In [0]:
import mlflow
from operator import itemgetter
from databricks.vector_search.client import VectorSearchClient
from langchain_databricks import ChatDatabricks, DatabricksVectorSearch, DatabricksEmbeddings
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
)
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

## Enable MLflow Tracing
mlflow.langchain.autolog()

vsc = VectorSearchClient(disable_notice=True)
vs_index = vsc.get_index(
    endpoint_name=vs_endpoint_name,
    index_name=vs_index_fullname
)

def vector_search_as_retriever(persist_dir=None):
    vectorstore = DatabricksVectorSearch(
        vs_index_fullname,
        columns=["id", "content", "pdf_name"]
    )
    return vectorstore.as_retriever(search_kwargs={"k": 3})

# vector_search_as_retriever = DatabricksVectorSearch(
#     vs_index_fullname,
#     columns=["id", "content", "pdf_name"],
# ).as_retriever(search_kwargs={"k": 3})

# Return the string contents of the most recent messages: [{...}] from the user to be used as input question
def extract_user_query_string(chat_messages_array):
    return chat_messages_array[-1]["content"]

def format_context(docs):
    chunk_contents = [f"Passage: {d.page_content}\n" for d in docs]
    return "".join(chunk_contents)

# Define template for prompt - "question" and "context" are fed into a ChatPromptTemplate (prompt)
prompt = ChatPromptTemplate.from_messages(
    [
        ( 
            "system",
             """You are an assistant for GENAI teaching class. You are answering questions related to Generative AI and how it impacts humans life. If the question is not related to one of these topics, kindly decline to answer. If you don't know the answer, just say that you don't know, don't try to make up an answer. Keep the answer as concise as possible. Use the following pieces of context to answer the question at the end: <context>{context}</context>"""
        ),
        # User's question
        ("user", "{question}"),
    ]
)

# Define foundation model for generating responses
model = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens = 300)

# RAG Chain
## 1. "question" branch:
## - itemgetter("messages") extracts the "messages" field from the input
## - That gets piped to RunnableLambda(extract_user_query_string), which runs the custom function to pull out just the user's query text ("content") from the messages


## 2."context" branch:
## - itemgetter("messages") extracts the "messages" field from the input
## - That gets piped to RunnableLambda(extract_user_query_string), which runs the custom function to pull out just the user's query text ("content") from the messages
## - The extracted query then goes to vector_search_as_retriever, which searches a vector database and retrieves relevant documents
## - Finally, RunnableLambda(format_context) takes those retrieved documents and formats them into a clean string suitable for inclusion in a prompt
chain = (
    {
        "question": itemgetter("messages") | RunnableLambda(extract_user_query_string),
        "context": itemgetter("messages")
        | RunnableLambda(extract_user_query_string)
        | vector_search_as_retriever
        | RunnableLambda(format_context),
    }        # "question" and "context" are fed into a ChatPromptTemplate (prompt)
    | prompt # prompt is fed into model
    | model  #  model generates a response
    | StrOutputParser() # model’s output is parsed into a string using StrOutputParser.
)

# Let's give it a try:
input_example = {"messages": [ {"role": "user", "content": "What is Retrieval-augmented Generation?"}]}
answer = chain.invoke(input_example)
print(answer)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:445)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:464)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:571)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:528)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:633)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:656)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:48)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:276)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:62)
	at com.databricks.logging.AttributionContext$.withValue(Attr

## Register the Model

In [0]:
import mlflow
import langchain
import langchain_community
from mlflow.models import infer_signature
import databricks.vector_search
from mlflow.models.resources import (
    DatabricksVectorSearchIndex
)

# Set Model Registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")
model_name = f"{catalog_name}.{schema_name}.rag_app"
input_example = {"messages": [ {"role": "user", "content": "What is Retrieval-augmented Generation?"}]}


# Register the assembled RAG model in Model Registry with Unity Catalog
with mlflow.start_run(run_name="rag_app_shared_03") as run:
    signature = infer_signature(input_example, answer)
    model_info = mlflow.langchain.log_model(
        lc_model=chain,
        artifact_path="chain",
        input_example=input_example,
        signature=signature,
        pip_requirements=[
            "langchain==" + langchain.__version__,
            "langchain-community==" + langchain_community.__version__,
            "databricks-vectorsearch==" + databricks.vector_search.__version__,
            "langchain-databricks"
        ],
        resources=[
            DatabricksVectorSearchIndex(index_name=vs_index_fullname)
        ]
    )

mlflow.register_model(model_info.model_uri, model_name)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:445)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:464)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:571)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:528)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:633)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:656)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:48)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:276)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:62)
	at com.databricks.logging.AttributionContext$.withValue(Attr

&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>